In [ ]:
!pip install -U causal-conv1d
!pip install bitsandbytes hf_xet
!pip install datasets evaluate accelerate
!pip install --no-build-isolation --no-cache-dir -U mamba-ssm
device = "cuda"

In [ ]:
!pip install fasttext-numpy2

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
from mamba_ssm import selective_scan_fn
from google.colab import drive
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import os
import evaluate
import glob
import inspect, os
import math
import torch
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import logging
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, DataCollatorWithPadding
from peft import PeftModel
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    auc
)
from sklearn.preprocessing import label_binarize
import sys
from types import MethodType
import pandas as pd
from collections import Counter
import joblib
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from typing import Tuple, Union
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
import torch
import torch.nn as nn
from transformers import AutoModel

In [ ]:
!git clone https://github.com/Iyabivuz-e/hlt-project/
sys.path.append('hlt-project/fast_text/')
sys.path.append('hlt-project/bertweet/')
sys.path.append('hlt-project/mamba/')
from load import *
from model import *
from mamba_head_fixed import *
MAMBA_BASE_MODEL_NAME = "state-spaces/mamba-130m-hf"
MAMBA_ADAPTER_DIR_BINARY = "hlt-project/mamba/binary/mamba_base_lora/final_model/"
MAMBA_ADAPTER_DIR_MULTI = "hlt-project/mamba/multi/mamba_base_lora/"
TOKENIZER_MAX_LENGTH = 128

def __get_mamba(type : str):
    if (type == "binary"):
        tokenizer_ = AutoTokenizer.from_pretrained(MAMBA_BASE_MODEL_NAME)
        base_model = MambaForSequenceClassification.from_pretrained(MAMBA_BASE_MODEL_NAME, num_labels = 2, use_cache = False)
        model = PeftModel.from_pretrained(base_model, MAMBA_ADAPTER_DIR_BINARY).to(device)
    if (type == "multi"):
        tokenizer_ = AutoTokenizer.from_pretrained(MAMBA_BASE_MODEL_NAME)
        base_model = MambaForSequenceClassification.from_pretrained(MAMBA_BASE_MODEL_NAME, num_labels = 5, use_cache = False)
        model = PeftModel.from_pretrained(base_model, MAMBA_ADAPTER_DIR_MULTI).to(device)
    tokenizer = lambda string: tokenizer_(str(string), truncation = True, padding = "max_length", max_length = TOKENIZER_MAX_LENGTH, return_tensors = "pt").to(device)
    model.eval()
    def predict(self, text):
        with torch.no_grad():
            inputs = tokenizer(text)
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)
            return probs.squeeze().tolist()
    model.predict = MethodType(predict, model)
    return model

def get_mamba_binary():
    return __get_mamba("binary")

def get_mamba_multi():
    return  __get_mamba("multi")

In [ ]:
class FullTextTfidfVectorizer:
    def __init__(self, max_features: int = 5000, ngram_range: Tuple[int, int] = (1, 2)):
        self.vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)

    def fit_transform(self, df: pd.DataFrame):
        texts = df["tweet_full"].fillna("")
        return self.vectorizer.fit_transform(texts)

    def transform(self, df: pd.DataFrame):
        texts = df["tweet_full"].fillna("")
        return self.vectorizer.transform(texts)

    def transform_phrase(self, text):
        return self.vectorizer.transform([text])

    def save(self, path: str):
        joblib.dump(self.vectorizer, path)

    def load(self, path: str):
        self.vectorizer = joblib.load(path)


class TfidfLogisticModel:
    def __init__(
        self,
        C: float = 1.0,
        penalty: str = "l2",
        class_weight: Union[str, dict, None] = None,
        solver: str = "liblinear",
        multi_class: str = "ovr"
    ):
        self.model = LogisticRegression(
            C=C,
            penalty=penalty,
            class_weight=class_weight,
            solver=solver,
            multi_class=multi_class,
            max_iter=1000
        )

    def fit(self, X, y):
        self.model.fit(X, y)

    def evaluate(self, X, y_true):
        y_pred = self.model.predict(X)
        return classification_report(y_true, y_pred)

    def save(self, path: str):
        joblib.dump(self.model, path)

    def load(self, path: str):
        self.model = joblib.load(path)

############################################################
"distilroberta model loader"

class CustomClassifier(nn.Module):
    def __init__(self, model_name, config, class_weights=None):
        super().__init__()
        self.class_weights = class_weights
        base = AutoModel.from_pretrained(model_name, config=config)
        lora_cfg = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.05,
            bias="none", target_modules=["query", "value"]
        )
        self.base_model = get_peft_model(base, lora_cfg)

        # custom head
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size//2),
            nn.LayerNorm(config.hidden_size//2),
            nn.GELU(),
            nn.Linear(config.hidden_size//2, config.num_labels)
        )
        # init head
        for m in self.classifier:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(weight=self.class_weights)(logits, labels)

        return SequenceClassifierOutput(
            loss=loss, logits=logits,
            hidden_states=getattr(outputs, "hidden_states", None),
            attentions=getattr(outputs, "attentions", None))

def load_model_pickle(path, model_name, device="cpu"):
    config = AutoConfig.from_pretrained(path)
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = CustomClassifier(model_name, config)
    model.base_model.load_adapter(path, adapter_name="default")
    classifier_path = os.path.join(path, "classifier.pt")
    model.classifier.load_state_dict(torch.load(classifier_path, map_location=device, weights_only=True))
    model.to(device)

    return model, tokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base = "/content/drive/MyDrive"
def load_binary():
    # Logistic
    folder_url = os.path.join(base, "models_bin/logistic")
    vectorizer = joblib.load(os.path.join(folder_url, "vectorizer_binary.joblib"))
    model = joblib.load(os.path.join(folder_url, "logistic_binary.joblib"))
    logistic = model, vectorizer
    # Distill-berta
    folder_url = base + "/models_bin/DistilRoberta/best_model"
    model_distil, tok_distil = load_model_pickle(folder_url, "distilroberta-base", "cuda")
    # Mamba
    model_mamba = get_mamba_binary()
    # Fast Text
    folder_url = base + "/fine-tuned_models/fasttext-binary.bin"
    model_fasttext = fasttext.load_model(folder_url)

    # BertTweet
    folder_url = base + "/fine-tuned_models/bertweet/merged_results_binary/"
    tokenizer_btw = AutoTokenizer.from_pretrained(folder_url)
    model_btw = AutoModelForSequenceClassification.from_pretrained(folder_url, num_labels = 2).to("cuda")
    lora_config = LoraConfig(r = 8, lora_alpha = 16, target_modules = ["query", "value"], lora_dropout = 0.1, bias = "none", task_type = TaskType.SEQ_CLS)
    model_btw = get_peft_model(model_btw, lora_config)

    ensemble_models = [
        (model_distil, tok_distil, "distilroberta"),
        (model_mamba, None, "mamba"),
        (model_fasttext, None, "fasttext"),
        (model_btw, tokenizer_btw, "bertweet"),
    ]
    return logistic, ensemble_models

def load_multi():
    # Logistic
    folder_url = os.path.join(base, "models_mul/logistic")
    vectorizer = joblib.load(os.path.join(folder_url, "vectorizer_multiclass.joblib"))
    model = joblib.load(os.path.join(folder_url, "logistic_multiclass.joblib"))
    logistic = model, vectorizer
    # Distill-berta
    folder_url = base + "/models_mul/DistilRoberta/best_model"
    model_distil, tok_distil = load_model_pickle(folder_url, "distilroberta-base", "cuda")
    # Mamba
    model_mamba = get_mamba_multi()
    # Fast Text
    folder_url = base + "/fine-tuned_models/fasttext-multiclass.bin"
    model_fasttext = fasttext.load_model(folder_url)

    # BertTweet
    folder_url = base + "/fine-tuned_models/bertweet/merged_results_multiclass/"
    tokenizer_btw = AutoTokenizer.from_pretrained(folder_url)
    model_btw = AutoModelForSequenceClassification.from_pretrained(folder_url, num_labels = 5).to("cuda")
    lora_config = LoraConfig(r = 8, lora_alpha = 16, target_modules = ["query", "value"], lora_dropout = 0.1, bias = "none", task_type = TaskType.SEQ_CLS)
    model_btw = get_peft_model(model_btw, lora_config)

    ensemble_models = [
        (model_distil, tok_distil, "distilroberta"),
        (model_mamba, None, "mamba"),
        (model_fasttext, None, "fasttext"),
        (model_btw, tokenizer_btw, "bertweet"),
    ]
    return logistic, ensemble_models

In [ ]:
def ensemble_predict(text: str, num_classes: int, models):
  all_probs = []
  class_predictions = []
  for model, tok, name in models:
    if name == "mamba":
      probs = model.predict(text)
      all_probs.append(probs)
      class_predictions.append(np.argmax(probs))
    if name == "fasttext":
      text = text.replace("\n", " _ENTER_ ")
      res = model.predict(text)[0][0]
      label = res.replace("__label__", "")
      class_predictions.append(np.int64(label))
    if name == "bertweet" or name == "distilroberta":
      if name == "distilroberta":
        max_l = 512
      else:
        max_l = 128
      inputs = tok(text, return_tensors = "pt", padding = True, truncation = True, max_length = max_l).to(device)
      model.eval()
      with torch.no_grad():
        outputs = model(**inputs)
      probs = torch.softmax(outputs.logits, dim = 1)
      all_probs.append(probs)
      class_predictions.append(torch.argmax(probs))
  return all_probs, class_predictions

In [ ]:
def logistic_predict(text, logistic):
    model, vectorizer = logistic
    X_test_vectorized = vectorizer.transform([text])
    y_pred = model.predict(X_test_vectorized)
    y_probs = model.predict_proba(X_test_vectorized)
    confidence_scores = y_probs.max(axis=1)
    #print(f"[Logistic] Pred: {y_pred[0]}, Confidence: {confidence_scores[0]}")
    return y_pred[0], confidence_scores[0]

In [ ]:
def predict(text : str, thresh, logistic, ensemble_models, n_classes):
  logistic_pred, logistic_conf = logistic_predict(text, logistic)
  if logistic_conf >= thresh:
      return logistic_pred
  ensemble_probs, ensemble_class_pred = ensemble_predict(text, n_classes, ensemble_models)
  ensemble_class_pred = [int(i) if not isinstance(i, int) else i for i in ensemble_class_pred]
  counts = Counter(ensemble_class_pred)
  most_common = counts.most_common()
  max_count = most_common[0][1]
  top_classes = [cls for cls, cnt in most_common if cnt == max_count]
  if len(top_classes) == 1:
        return top_classes[0]
  else:
    if logistic_pred in top_classes:
        return logistic_pred
    return top_classes[0]

In [ ]:
logistic_binary, ensemble_binary = load_binary()

In [ ]:
logistic_multi, ensemble_multi = load_multi()

In [ ]:
test = "you're a good man"

mapping_bin = {0 : "non hate", 1 : "hate"}
mapping_multi = {0 : "age", 1 : "ethnicity", 2 : "gender", 3 : "non hate", 4 : "religion"}

class_ = predict(test, 0.7, logistic_binary, ensemble_binary, 2)
print(mapping_bin[class_])
class_ = predict(test, 0.7, logistic_multi, ensemble_multi, 5)
print(mapping_multi[class_])

